# Prepping Data Challenge - Week 10

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
transaction_data = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_10_Transaction_Data.csv')
loyalty_table = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_10_Loyalty_Table.csv')
product_table = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_10_Product_Table.csv')

In [3]:
transaction_data.head()

,Transaction_Date,Transanction_Number,Product_ID,Cash_or_Card,Loyalty_Number,Sales_Before_Discount
0,"Sat, January 02, 2021",20121001,Bar-Ocean_Mist-1x,1,1004721.0,6.00
1,"Sat, January 02, 2021",20121001,Liquid-Rose_Garden-0.5L,1,1004721.0,14.10
2,"Sat, January 02, 2021",20121002,Bar-Citrus_Breeze-4x,2,1009280.0,8.12
3,"Sat, January 02, 2021",20121002,Liquid-Coconut_Dream-0.5L,2,1009280.0,12.36
4,"Sat, January 02, 2021",20121003,Bar-Ocean_Mist-4x,1,1009022.0,13.95


In [4]:
loyalty_table.head()

,Loyalty_Number,Customer_Name,Loyalty_Tier,Loyalty_Discount
0,1000012,"trimmill, leeanne",Bronz,NaN
1,1000026,"kobierski, teador",NaN,NaN
2,1000028,"plues, jenelle",Bronz,NaN
3,1000032,"firmager, gabriell",Bronz,NaN
4,1000038,"chiles, nicolea",NaN,NaN


In [5]:
product_table.head()

,Product_Type,Product_Scent,Pack_Size,Product_Size,Unit_Cost,Selling_Price
0,Bar,Lavender Fields,1x,NaN,1.25,1.77
1,Bar,Citrus Breeze,1x,NaN,0.75,0.81
2,Bar,Ocean Mist,1x,NaN,0.66,1.20
3,Bar,Fresh Rain,1x,NaN,0.94,1.61
4,Bar,Rose Garden,1x,NaN,1.55,2.45


In [6]:
print(product_table.shape)
print(loyalty_table.shape)
print(transaction_data.shape)

(60, 6)
(9789, 4)
(105495, 6)


## Challenges

### 1. Filter to the last 2 years (2023 + 2024)

In [7]:
# Turn transaction string to Datetime
transaction_data['Date'] = pd.to_datetime(transaction_data['Transaction_Date'], format="%a, %B %d, %Y")

# Filter for the last two years
recent_transactions = transaction_data[transaction_data['Date'].dt.year >= 2023].copy()

### 2. Create empty rows for days where the shop was closed

In [8]:
# Use merge to add the missing Dates
all_dates_23_24 = pd.DataFrame({'Date' : pd.date_range(np.datetime64('2023-01-01'), recent_transactions['Date'].max())})
recent_transactions = all_dates_23_24.merge(recent_transactions, on='Date', how='left')

In [9]:
recent_transactions

,Date,Transaction_Date,Transanction_Number,Product_ID,Cash_or_Card,Loyalty_Number,Sales_Before_Discount
0,2023-01-01,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03,"Tue, January 03, 2023",30123001.0,Liquid-Sandalwood_Spice-0.25L,2.0,1005245.0,8.50
3,2023-01-03,"Tue, January 03, 2023",30123001.0,Liquid-Vanilla_Bean-0.5L,2.0,1005245.0,14.70
4,2023-01-03,"Tue, January 03, 2023",30123002.0,Liquid-Sandalwood_Spice-1L,1.0,1007270.0,13.19
...,...,...,...,...,...,...,...
39334,2024-03-06,"Wed, March 06, 2024",60324062.0,Bar-Sandalwood_Spice-4x,2.0,1001139.0,31.96
39335,2024-03-06,"Wed, March 06, 2024",60324062.0,Liquid-Cedarwood_Forest-0.25L,2.0,1001139.0,7.80
39336,2024-03-06,"Wed, March 06, 2024",60324063.0,Liquid-Citrus_Breeze-0.5L,1.0,1007693.0,2.48
39337,2024-03-06,"Wed, March 06, 2024",60324063.0,Liquid-Sandalwood_Spice-0.5L,1.0,1007693.0,47.80


### 3. Update the Cash or Card Field

In [10]:
recent_transactions['Cash_or_Card'] = recent_transactions['Cash_or_Card'].map({1 : "Card", 2 : "Cash"})

### 4. Join the Product Table

In [11]:
# Create a Product_ID column by using the np.where method
product_table['Product_ID'] = product_table['Product_Type'] + '-' + product_table['Product_Scent'] + '-' + np.where(product_table['Product_Type'] == 'Bar', product_table['Pack_Size'], product_table['Product_Size'])

# We also need to adjust the string formatting slightly
product_table['Product_ID'] = product_table['Product_ID'].apply(lambda x: x.replace(' ', '_'))

In [12]:
transactions = recent_transactions.copy().merge(product_table, on='Product_ID', how='left')

### 5. Calculate the quantity of each sale

In [13]:
transactions['Quantity'] = transactions['Sales_Before_Discount'] / transactions['Selling_Price']

### 6. Transform the loyalty table

#### 6.1. Change the name format

In [14]:
loyalty_table['Customer_Name'] = loyalty_table['Customer_Name'].apply(lambda x: str(x).split(', ')[1].capitalize() + ' ' + str(x).split(', ')[0].capitalize())

#### 6.2. Group into bronze, silver, and gold tier

In [15]:
loyalty_table['Loyalty_Tier'].unique()

array(['Bronz', nan, 'Bronze', 'bronze', 'Goald', 'Gold', 'gold',
       'Silver', 'Sliver', 'silver'], dtype=object)

In [20]:
nans_before_mapping = loyalty_table['Loyalty_Tier'].isna().sum()

In [17]:
loyalty_table['Loyalty_Tier'] =  loyalty_table['Loyalty_Tier'].map({'Bronz' : 'Bronze', 
                                                                    'bronze' : 'Bronze', 
                                                                    'Bronze' : 'Bronze',
                                                                    'Goald' : 'Gold', 
                                                                    'gold' : 'Gold',
                                                                    'Gold' : 'Gold',
                                                                    'Sliver' : 'Silver', 
                                                                    'silver' : 'Silver',
                                                                    'Silver' : 'Silver'})

In [18]:
loyalty_table['Loyalty_Tier'].unique()

array(['Bronze', nan, 'Gold', 'Silver'], dtype=object)

In [21]:
assert nans_before_mapping == loyalty_table['Loyalty_Tier'].isna().sum()

#### 6.3. Change the discount field to numeric

In [30]:
loyalty_table['Loyalty_Discount'] = loyalty_table['Loyalty_Discount'].fillna('0%').apply(lambda x: int(x[:-1]) / 100)

### 7. Join the loyalty table

In [35]:
transactions = transactions.merge(loyalty_table, on='Loyalty_Number')

### 8. Create a `Sales_After_Discount` field to apply the `Loyalty_Discount` for transactions with a `Loyalty_Number`

In [38]:
transactions['Sales_After_Discount'] = transactions['Sales_Before_Discount'] * (1-transactions['Loyalty_Discount'])

### 9. Calculate the profit

In [41]:
transactions['Profit'] = transactions['Sales_After_Discount'] - (transactions['Unit_Cost'] * transactions['Quantity'])

### 10. Format the column Names

In [45]:
transactions.columns = [x.replace('_', ' ') for x in transactions.columns]

### 11. Only keep useful columns

In [56]:
useful_columns = 'Date, Transanction Number, Product Type, Product Scent, Pack Size, Product Size, Cash or Card, Loyalty Number, Customer Name, Loyalty Tier, Loyalty Discount, Quantity, Sales Before Discount, Sales After Discount, Profit'.split(', ')

In [58]:
final_table = transactions.loc[:, useful_columns].copy() 

In [59]:
useful_columns_corrected = 'Date, Transaction Number, Product Type, Product Scent, Pack Size, Product Size, Cash or Card, Loyalty Number, Customer Name, Loyalty Tier, Loyalty Discount, Quantity, Sales Before Discount, Sales After Discount, Profit'.split(', ')

In [60]:
final_table.columns = useful_columns_corrected

## Save Output Data

In [61]:
final_table.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/10_transactions_with_profits.csv')